# Demo 2: Object Detection

**Before you start: go to Runtime → Change runtime type and select GPU.**

Image classification answers *what* is in an image. Object detection answers 
*what* and *where* — it finds every object in a scene, draws a bounding box 
around each one, and assigns a class label and confidence score.

This is a much harder problem than classification. A single image might contain 
dozens of objects at different scales, overlapping each other, partially hidden. 
The model has to find all of them simultaneously.

Object detection is used in autonomous vehicles (detect pedestrians, cars, 
traffic signs), medical imaging (locate tumours or lesions), retail (count 
products on shelves), wildlife monitoring (count animals in aerial images), 
and many more applications.

In this demo you will:
1. Run a state-of-the-art detector on example images
2. Explore how confidence thresholds affect what gets detected
3. Try it on images of your own choice
4. See what kinds of problems detection is — and is not — suited to

## Setup

In [ ]:
import torch
from torchvision import models, transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import urllib.request
import colorsys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Loading a pretrained detector

We will use **Faster R-CNN** with a ResNet-50 backbone — one of the most 
widely used object detection architectures. It was trained on **COCO** 
(Common Objects in Context), a dataset of 120,000 images with 80 categories 
of everyday objects annotated with bounding boxes.

The 80 COCO categories include people, animals, vehicles, furniture, food, 
sports equipment, and common household objects.

In [ ]:
# Load Faster R-CNN pretrained on COCO
weights = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
model   = fasterrcnn_resnet50_fpn_v2(weights=weights)
model   = model.to(device)
model.eval()

# The preprocessing transform recommended for this model
preprocess = weights.transforms()

# COCO category names
COCO_CLASSES = weights.meta['categories']
print(f'Model loaded. Detects {len(COCO_CLASSES)} categories:')
print(', '.join(COCO_CLASSES))

In [ ]:
# Generate a distinct colour for each category
def get_colors(n):
    colors = []
    for i in range(n):
        hue = i / n
        rgb = colorsys.hsv_to_rgb(hue, 0.8, 0.95)
        colors.append(rgb)
    return colors

CLASS_COLORS = get_colors(len(COCO_CLASSES))


def detect_and_display(image_path, confidence_threshold=0.5, max_boxes=20):
    """
    Run object detection on an image and display the results.
    
    Args:
        image_path:            path to image file or URL
        confidence_threshold:  only show detections above this confidence (0–1)
        max_boxes:             maximum number of boxes to display
    """
    # Load image
    if image_path.startswith('http'):
        urllib.request.urlretrieve(image_path, 'temp_detect.jpg')
        image_path = 'temp_detect.jpg'
    image = Image.open(image_path).convert('RGB')
    
    # Preprocess and run model
    tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        predictions = model(tensor)[0]
    
    # Filter by confidence threshold
    boxes   = predictions['boxes'].cpu().numpy()
    labels  = predictions['labels'].cpu().numpy()
    scores  = predictions['scores'].cpu().numpy()
    
    mask   = scores >= confidence_threshold
    boxes  = boxes[mask][:max_boxes]
    labels = labels[mask][:max_boxes]
    scores = scores[mask][:max_boxes]
    
    # Display
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(image)
    
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        class_name = COCO_CLASSES[label]
        color      = CLASS_COLORS[label % len(CLASS_COLORS)]
        
        # Draw bounding box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Draw label
        ax.text(
            x1, y1 - 4,
            f'{class_name} {score:.2f}',
            color='white', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8)
        )
    
    n_detected = len(boxes)
    ax.set_title(
        f'{n_detected} object{"s" if n_detected != 1 else ""} detected '
        f'(confidence ≥ {confidence_threshold:.0%})',
        fontsize=13
    )
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Print summary
    if n_detected > 0:
        from collections import Counter
        counts = Counter(COCO_CLASSES[l] for l in labels)
        print('Detected:', ', '.join(f'{v}x {k}' for k, v in sorted(counts.items())))
    else:
        print('No objects detected above the confidence threshold.')
    
    return boxes, labels, scores

print('Functions defined. Ready to detect objects.')

## 2. Running the detector

In [ ]:
# A busy street scene
detect_and_display(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Kitano_Street_Kobe01s5s4110.jpg/640px-Kitano_Street_Kobe01s5s4110.jpg',
    confidence_threshold=0.5
)

In [ ]:
# A kitchen scene — lots of everyday objects
detect_and_display(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/6/6d/Good_Food_Display_-_NCI_Visuals_Online.jpg/640px-Good_Food_Display_-_NCI_Visuals_Online.jpg',
    confidence_threshold=0.5
)

## 3. The confidence threshold

Every detection comes with a **confidence score** between 0 and 1 — 
how certain the model is that this box really contains the predicted object.

The threshold is a design choice that trades off **precision** against **recall**:
- **High threshold** (e.g. 0.9): only very confident detections shown — fewer boxes, but more likely to be correct
- **Low threshold** (e.g. 0.2): many detections shown — catches more real objects, but also more false positives

The right threshold depends entirely on your application. A self-driving car 
prefers to miss nothing (low threshold). A photo tagging app prefers to only 
show confident labels (high threshold).

Let's see how the threshold affects what gets detected:

In [ ]:
street_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Kitano_Street_Kobe01s5s4110.jpg/640px-Kitano_Street_Kobe01s5s4110.jpg'

print('=== High threshold (0.8) — only very confident detections ===')
detect_and_display(street_url, confidence_threshold=0.8)

print('\n=== Low threshold (0.3) — many detections, including uncertain ones ===')
detect_and_display(street_url, confidence_threshold=0.3)

## 4. Try it on your own image

In [ ]:
# Replace with any image URL you want to try
my_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/38/Shopping_Center_Interior.jpg/640px-Shopping_Center_Interior.jpg'

detect_and_display(my_url, confidence_threshold=0.5)

In [ ]:
# Or upload your own image
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    detect_and_display(filename, confidence_threshold=0.5)

## 5. What the model cannot do

This model was trained on 80 COCO categories. It will not detect anything 
outside those categories — no matter how clearly that object appears in the image. 
A model trained to detect cars cannot detect, say, a specific type of tumour 
in a medical scan — that requires training on medical data.

This is a fundamental property of supervised learning: **the model can only 
recognise what it has been trained to recognise.** Every project that uses 
detection will either:
- Use a pretrained model on categories it already knows, or
- Fine-tune a pretrained model on new categories using custom annotated data

Let's deliberately try an image the model should struggle with:

In [ ]:
# Microscopy image — very different from natural images
detect_and_display(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/24701-nature-natural-beauty.jpg/640px-24701-nature-natural-beauty.jpg',
    confidence_threshold=0.3   # low threshold to see even uncertain detections
)
print()
print('The model tries to match what it sees to COCO categories it knows.')
print('On out-of-distribution images, predictions are unreliable.')

## 6. What can you do with this?

Object detection opens up problems that classification cannot solve — any task 
where you need to find *and* locate objects, or where multiple objects appear 
in a single image. Examples:

- **Counting objects** — count cells in a microscopy image, animals in a wildlife photo, or products on a shelf
- **Monitoring** — detect people or vehicles in surveillance footage
- **Sports analysis** — track players and the ball during a game
- **Agriculture** — detect and count fruit on trees from drone images
- **Document analysis** — locate text regions, figures, or tables in scanned documents

Note that for custom categories you would need to annotate your own bounding box 
data — which is significantly more effort than collecting classification labels. 
Tools like [Label Studio](https://labelstud.io/) or [Roboflow](https://roboflow.com/) 
make this easier.

---

### Think about your project

1. Is there a problem you are interested in where location matters — where it is not enough to just know *what* is in the image?
2. Would your data already have bounding box annotations, or would you need to create them?
3. How many objects would typically appear in a single image in your domain?